# 27 — Semisupervised Trainability Audit

## Objetivo

Determinar si el conjunto experimental `L ∪ U` materializado en el notebook 26 puede sostener científicamente:

- un baseline supervisado;
- aprendizaje semisupervisado convencional;
- aprendizaje Positive–Unlabeled (PU);
- o si primero debe recuperarse más evidencia etiquetada de los datasets existentes.

Este notebook **no entrena modelos**.

La decisión se basa en cuatro condiciones:

1. soporte observado de clases en `L`;
2. número de grupos etiquetados;
3. posibilidad real de reservar evaluación;
4. cambio de dominio entre `L` y `U`.

Además, se vuelve a inspeccionar la familia de artefactos Catching para localizar sitios con etiquetas negativas ya existentes cuya X aún no haya sido recuperada.

Esto permite distinguir entre:

```text
NO HAY DATOS
```

y:

```text
LAS ETIQUETAS EXISTEN, PERO TODAVÍA FALTA RECUPERAR X COMPATIBLE
```

La segunda situación es metodológicamente mucho más útil para la tesis.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import warnings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_DIR = PROJECT_DIR / "data" / "semisupervised"
SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
SEMISUP_INTERIM_DIR = SEMISUP_DIR / "interim"
TARGET_DIR = SEMISUP_INTERIM_DIR / "targets"

SEMISUP_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

FEATURES = [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar master y registros del notebook 26

In [2]:
master = pd.read_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset.csv"
)

membership = pd.read_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_target_membership_long.csv"
)

split_summary = pd.read_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_target_split_summary.csv"
)

labeled_balance = pd.read_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_labeled_balance.csv"
)

domain_registry = pd.read_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_domain_registry.csv"
)

print("Master:", master.shape)
print("Membership:", membership.shape)

display(split_summary)
display(labeled_balance)
display(domain_registry)


Master: (1002, 22)
Membership: (4008, 9)


,target,EXCLUDED_INCOMPLETE_X,L,U,EXCLUDED_INVALID_Y,usable_rows
0,H1_violation,2,2,998,0,1000
1,H4_violation,2,2,998,0,1000
2,H5_violation,2,2,998,0,1000
3,H6_violation,2,2,998,0,1000


,target,n_labeled,n_0,n_1,n_classes
0,H1_violation,2,0,2,1
1,H4_violation,2,0,2,1
2,H5_violation,2,0,2,1
3,H6_violation,2,0,2,1


,source_dataset,domain,ssl_role,n_rows,scientific_warning
0,Catching_UX_Flaws,WEB_DOM,L,2,Very small labeled seed; two sites only.
1,UICrit_RICO,ANDROID_UI,U,1000,Unlabeled pool comes from a different UI platform/domain. SSL assumptions must be audited before training.


## 3. Integridad mínima de la entrada

In [3]:
input_checks = []

input_checks.append({
    "check": "MASTER_NOT_EMPTY",
    "passed": len(master) > 0,
    "observed": len(master)
})

input_checks.append({
    "check": "EXPECTED_FEATURES_PRESENT",
    "passed": set(FEATURES).issubset(master.columns),
    "observed": " | ".join(
        [
            feature
            for feature in FEATURES
            if feature in master.columns
        ]
    )
})

input_checks.append({
    "check": "EXPECTED_TARGETS_PRESENT",
    "passed": set(TARGETS).issubset(master.columns),
    "observed": " | ".join(
        [
            target
            for target in TARGETS
            if target in master.columns
        ]
    )
})

input_checks.append({
    "check": "DUPLICATE_MASTER_ID",
    "passed": (
        master[
            "master_instance_id"
        ].duplicated().sum()
        ==
        0
    ),
    "observed": int(
        master[
            "master_instance_id"
        ].duplicated().sum()
    )
})

input_checks = pd.DataFrame(
    input_checks
)

display(input_checks)

assert input_checks["passed"].all()


,check,passed,observed
0,MASTER_NOT_EMPTY,True,1002
1,EXPECTED_FEATURES_PRESENT,True,structure_depth | interactive_control_count | interactive_label_ratio
2,EXPECTED_TARGETS_PRESENT,True,H1_violation | H4_violation | H5_violation | H6_violation
3,DUPLICATE_MASTER_ID,True,0


## 4. Soporte observado por target

Para clasificación binaria ordinaria, `L` debe contener ambas clases observadas.

No se considera válido inferir la clase faltante desde `U`.


In [4]:
class_support_records = []

for target in TARGETS:

    target_l = membership[
        (
            membership["target"] == target
        )
        &
        (
            membership["ssl_role"] == "L"
        )
    ]

    labeled_ids = target_l[
        "master_instance_id"
    ].tolist()

    values = (
        master
        .set_index("master_instance_id")
        .loc[
            labeled_ids,
            target
        ]
        if labeled_ids
        else pd.Series(dtype=float)
    )

    groups = (
        target_l[
            "group_id"
        ]
        .dropna()
        .astype(str)
        .nunique()
    )

    class_support_records.append({
        "target": target,
        "n_L": len(target_l),
        "n_groups_L": int(groups),
        "n_0": int((values == 0).sum()),
        "n_1": int((values == 1).sum()),
        "n_classes": int(
            values.dropna().nunique()
        ),
        "both_classes_observed": (
            values.dropna().nunique()
            >=
            2
        )
    })

class_support = pd.DataFrame(
    class_support_records
)

display(class_support)


,target,n_L,n_groups_L,n_0,n_1,n_classes,both_classes_observed
0,H1_violation,2,2,0,2,1,False
1,H4_violation,2,2,0,2,1,False
2,H5_violation,2,2,0,2,1,False
3,H6_violation,2,2,0,2,1,False


## 5. ¿Puede existir un test etiquetado válido?

Para evaluar precision, recall, F1 o ROC/PR de un clasificador binario se necesita un conjunto de evaluación con ground truth relevante.

Con `L` de una sola clase, no puede construirse un test que mida discriminación 0 vs 1.

Esta comprobación no usa un umbral arbitrario de tamaño: evalúa una condición lógica de identificabilidad.


In [5]:
evaluation_feasibility = class_support[
    [
        "target",
        "n_L",
        "n_groups_L",
        "n_0",
        "n_1",
        "n_classes"
    ]
].copy()

evaluation_feasibility[
    "binary_discrimination_test_possible"
] = (
    evaluation_feasibility[
        "n_classes"
    ]
    >=
    2
)

evaluation_feasibility[
    "grouped_holdout_with_both_classes_possible"
] = False

evaluation_feasibility[
    "reason"
] = np.where(
    evaluation_feasibility[
        "n_classes"
    ]
    <
    2,
    (
        "Only one observed class exists in L; "
        "a binary holdout cannot contain both classes."
    ),
    (
        "Class support exists; grouped split still "
        "requires group-level feasibility analysis."
    )
)

display(evaluation_feasibility)


,target,n_L,n_groups_L,n_0,n_1,n_classes,binary_discrimination_test_possible,grouped_holdout_with_both_classes_possible,reason
0,H1_violation,2,2,0,2,1,False,False,Only one observed class exists in L; a binary holdout cannot contain both classes.
1,H4_violation,2,2,0,2,1,False,False,Only one observed class exists in L; a binary holdout cannot contain both classes.
2,H5_violation,2,2,0,2,1,False,False,Only one observed class exists in L; a binary holdout cannot contain both classes.
3,H6_violation,2,2,0,2,1,False,False,Only one observed class exists in L; a binary holdout cannot contain both classes.


## 6. Forma Positive–Unlabeled (PU)

PU learning parte de:

- ejemplos conocidos positivos `P`;
- un conjunto `U` que mezcla positivos y negativos desconocidos.

El hecho de que nuestros datos tengan forma `P + U` **no significa automáticamente que PU sea científicamente evaluable**.

Se auditan por separado:

- existencia de positivos;
- existencia de U;
- ground truth con negativos para evaluación;
- correspondencia de dominio.


In [6]:
pu_records = []

for target in TARGETS:

    row = class_support[
        class_support["target"] == target
    ].iloc[0]

    target_u = membership[
        (
            membership["target"] == target
        )
        &
        (
            membership["ssl_role"] == "U"
        )
    ]

    positive_seed_exists = (
        row["n_1"] > 0
    )

    observed_negative_exists = (
        row["n_0"] > 0
    )

    unlabeled_pool_exists = (
        len(target_u) > 0
    )

    pu_form_present = (
        positive_seed_exists
        and
        not observed_negative_exists
        and
        unlabeled_pool_exists
    )

    pu_records.append({
        "target": target,
        "n_positive_seed": int(row["n_1"]),
        "n_observed_negative": int(row["n_0"]),
        "n_unlabeled": len(target_u),
        "pu_form_present": pu_form_present,
        "observed_negative_test_available": (
            observed_negative_exists
        )
    })

pu_audit = pd.DataFrame(
    pu_records
)

display(pu_audit)


,target,n_positive_seed,n_observed_negative,n_unlabeled,pu_form_present,observed_negative_test_available
0,H1_violation,2,0,998,True,False
1,H4_violation,2,0,998,True,False
2,H5_violation,2,0,998,True,False
3,H6_violation,2,0,998,True,False


## 7. Cambio de dominio `WEB_DOM → ANDROID_UI`

El `L` actual proviene de Catching/Web DOM y el `U` actual de RICO/Android UI hierarchy.

Se realiza un análisis **descriptivo**, no una prueba inferencial, porque `L` solo tiene dos sitios.

Para cada feature se reporta:

- rango de U;
- mediana e IQR de U;
- los dos valores de L;
- z-score de cada valor L respecto a la distribución U;
- si L cae dentro del rango de U.

Esto no demuestra equivalencia de dominio.


In [7]:
L_master = master[
    master[
        "source_dataset"
    ]
    ==
    "Catching_UX_Flaws"
].copy()

U_master = master[
    master[
        "source_dataset"
    ]
    ==
    "UICrit_RICO"
].copy()

domain_shift_records = []

for feature in FEATURES:

    l_values = pd.to_numeric(
        L_master[feature],
        errors="coerce"
    ).dropna()

    u_values = pd.to_numeric(
        U_master[feature],
        errors="coerce"
    ).dropna()

    u_mean = float(
        u_values.mean()
    )

    u_std = float(
        u_values.std(ddof=0)
    )

    u_min = float(
        u_values.min()
    )

    u_q1 = float(
        u_values.quantile(0.25)
    )

    u_median = float(
        u_values.median()
    )

    u_q3 = float(
        u_values.quantile(0.75)
    )

    u_max = float(
        u_values.max()
    )

    record = {
        "feature": feature,
        "n_L": len(l_values),
        "n_U": len(u_values),
        "U_min": u_min,
        "U_q1": u_q1,
        "U_median": u_median,
        "U_q3": u_q3,
        "U_max": u_max,
        "L_values": " | ".join(
            f"{v:.6g}"
            for v in l_values.tolist()
        ),
        "all_L_inside_U_range": bool(
            (
                (l_values >= u_min)
                &
                (l_values <= u_max)
            ).all()
        )
        if len(l_values) > 0
        else False
    }

    for idx, value in enumerate(
        l_values.tolist(),
        start=1
    ):

        z = (
            (value - u_mean)
            /
            u_std
            if u_std > 0
            else np.nan
        )

        record[
            f"L{idx}_z_vs_U"
        ] = float(z)

    domain_shift_records.append(record)

domain_shift_descriptive = pd.DataFrame(
    domain_shift_records
)

display(domain_shift_descriptive)


,feature,n_L,n_U,U_min,U_q1,U_median,U_q3,U_max,L_values,all_L_inside_U_range,L1_z_vs_U,L2_z_vs_U
0,structure_depth,2,1000,4.0,9.000000,11.000000,14.000000,25.0,8 | 15,True,-0.936686,0.842153
1,interactive_control_count,2,1000,0.0,4.000000,8.000000,13.000000,80.0,23 | 585,False,1.483074,65.558076
2,interactive_label_ratio,2,998,0.0,0.142857,0.333333,0.588141,1.0,0.73913 | 0.842735,True,1.268181,1.636118


## 8. Distancia multivariada descriptiva

Se estandarizan las tres X usando **solo U** y se calcula, para cada sitio de Catching, la distancia euclidiana al vecino más cercano de U.

Esta medida se usa únicamente como señal descriptiva de solapamiento; no como prueba de identidad entre dominios.


In [8]:
U_complete = U_master[
    FEATURES
].apply(
    pd.to_numeric,
    errors="coerce"
).dropna()

L_complete = L_master[
    FEATURES
].apply(
    pd.to_numeric,
    errors="coerce"
).dropna()

u_mean = U_complete.mean()
u_std = U_complete.std(ddof=0).replace(0, np.nan)

U_z = (
    (U_complete - u_mean)
    /
    u_std
).dropna()

L_z = (
    (L_complete - u_mean)
    /
    u_std
).dropna()

nn_records = []

for idx, row in L_z.iterrows():

    diff = (
        U_z
        -
        row.values
    )

    distances = np.sqrt(
        (
            diff ** 2
        )
        .sum(axis=1)
    )

    nearest_idx = distances.idxmin()

    nn_records.append({
        "L_master_index": idx,
        "L_master_instance_id":
            L_master.loc[
                idx,
                "master_instance_id"
            ],
        "nearest_U_master_index":
            nearest_idx,
        "nearest_U_master_instance_id":
            U_master.loc[
                nearest_idx,
                "master_instance_id"
            ],
        "standardized_euclidean_distance":
            float(
                distances.loc[
                    nearest_idx
                ]
            )
    })

nearest_neighbor_domain_audit = pd.DataFrame(
    nn_records
)

display(
    nearest_neighbor_domain_audit
)


,L_master_index,L_master_instance_id,nearest_U_master_index,nearest_U_master_instance_id,standardized_euclidean_distance
0,0,catching_ux_flaws__startbootstrap_3_col_portfolio,820,uicrit_rico__3113,0.614446
1,1,catching_ux_flaws__startbootstrap_sb_admin_2,980,uicrit_rico__46664,57.658694


## 9. Auditoría de supuestos para SSL convencional

El Self-Training estándar necesita un clasificador supervisado inicial capaz de distinguir clases.

Con una sola clase observada, ese clasificador binario inicial no es identificable.


In [9]:
ssl_readiness_records = []

for _, row in class_support.iterrows():

    target = row["target"]

    has_two_classes = bool(
        row["both_classes_observed"]
    )

    n_groups = int(
        row["n_groups_L"]
    )

    target_u = membership[
        (
            membership["target"] == target
        )
        &
        (
            membership["ssl_role"] == "U"
        )
    ]

    has_u = len(target_u) > 0

    ssl_readiness_records.append({
        "target": target,
        "supervised_binary_seed_identifiable":
            has_two_classes,
        "unlabeled_pool_available":
            has_u,
        "labeled_groups":
            n_groups,
        "conventional_self_training_ready":
            (
                has_two_classes
                and
                has_u
            ),
        "reason":
            (
                "Observed L has only one class."
                if not has_two_classes
                else "Binary seed is identifiable."
            )
    })

ssl_readiness = pd.DataFrame(
    ssl_readiness_records
)

display(
    ssl_readiness
)


,target,supervised_binary_seed_identifiable,unlabeled_pool_available,labeled_groups,conventional_self_training_ready,reason
0,H1_violation,False,True,2,False,Observed L has only one class.
1,H4_violation,False,True,2,False,Observed L has only one class.
2,H5_violation,False,True,2,False,Observed L has only one class.
3,H6_violation,False,True,2,False,Observed L has only one class.


## 10. Buscar artefactos Catching con etiquetas más amplias

Los notebooks anteriores mostraron que el dataset Catching completo tenía más sitios etiquetados que los dos repositorios `CONFIRMED_STRONG`.

Aquí se localizan automáticamente CSVs relacionados con Catching y se buscan tablas con:

- más de dos filas;
- columnas H1/H4/H5/H6;
- posibles clases negativas.

El objetivo no es usarlas automáticamente, sino determinar si existe una ruta de recuperación de X dentro de los datasets ya disponibles.


In [10]:
SEARCH_ROOTS = [
    PUBLIC_INTERIM_DIR,
    REPORTS_DIR
]

catching_csvs = []

seen = set()

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*.csv"):

        if "catch" not in path.name.lower():
            continue

        key = str(path.resolve())

        if key in seen:
            continue

        seen.add(key)

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        catching_csvs.append({
            "path":
                str(
                    path.relative_to(
                        PROJECT_DIR
                    )
                ),
            "rows":
                len(df),
            "columns":
                len(df.columns),
            "target_columns":
                " | ".join(
                    [
                        target
                        for target in TARGETS
                        if target in df.columns
                    ]
                ),
            "column_names":
                " | ".join(
                    df.columns.astype(str)
                )
        })

catching_artifact_inventory = pd.DataFrame(
    catching_csvs
)

display(
    catching_artifact_inventory
)


,path,rows,columns,target_columns,column_names
0,data/public_v2/interim/catching_labels_consensus_long.csv,120,6,,site | heuristic | n_votes | positive_votes | consensus | max_severity
1,data/public_v2/interim/catching_static_page_features.csv,15,14,,site | repo_full_name | relative_path | static_element_count | static_dom_max_depth | static_interactive_control_count | static_labeled_control_count | static_control_label_proxy_ratio | static_form_control_count | static_native_validated_control_count | static_native_validation_ratio | class_token_count | unique_class_count | static_class_reuse_ratio
2,data/public_v2/interim/catching_static_site_features.csv,2,12,,site | repo_full_name | static_page_count | static_mean_element_count | static_max_element_count | static_mean_dom_depth | static_max_dom_depth | static_total_interactive_controls | static_control_label_proxy_ratio | static_total_form_controls | static_native_validation_ratio | static_mean_class_reuse_ratio
3,data/public_v2/interim/catching_targets.csv,30,8,H1_violation | H4_violation | H5_violation | H6_violation,site | source_dataset | label_source | label_level | H1_violation | H4_violation | H5_violation | H6_violation
4,data/public_v2/interim/catching_static_validation_dataset.csv,2,23,H1_violation | H4_violation | H5_violation | H6_violation,site | repo_full_name | static_page_count | static_mean_element_count | static_max_element_count | static_mean_dom_depth | static_max_dom_depth | static_total_interactive_controls | static_control_label_proxy_ratio | static_total_form_controls | static_native_validation_ratio | static_mean_class_reuse_ratio | commit_sha | label_source | label_level | H1_violation | H4_violation | H5_violation | H6_violation | source_dataset | feature_modality | primary_training | intended_use
5,data/public_v2/interim/catching_labels_evaluator.csv,358,5,,site | evaluation | heuristic | violation | severity
6,data/public_v2/interim/catching_direct_validation_registry.csv,2,16,H1_violation | H4_violation | H5_violation | H6_violation,site | repo_full_name | commit_sha | branch | license_name | local_path | source_dataset | label_source | label_level | H1_violation | H4_violation | H5_violation | H6_violation | repository_quality | primary_training | intended_use
7,reports/tables/catching_source_recovery_status.csv,30,7,,site | normalized_site | candidate_count | exact_match_count | recovery_status | confirmed_repo | evidence
8,reports/tables/catching_single_candidate_validation.csv,11,10,,site | repo_full_name | clone_ok | source_files | snippets_tested | matched_snippets | matched_evaluations | matched_files | error | validation_status
9,reports/tables/catching_confirmed_extension_summary.csv,17,3,,repo_full_name | extension | file_count


## 11. Identificar posibles tablas Catching con clases negativas

In [11]:
candidate_label_tables = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*.csv"):

        if "catch" not in path.name.lower():
            continue

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        existing_targets = [
            target
            for target in TARGETS
            if target in df.columns
        ]

        if len(existing_targets) == 0:
            continue

        negative_cells = 0
        positive_cells = 0

        for target in existing_targets:

            values = pd.to_numeric(
                df[target],
                errors="coerce"
            )

            negative_cells += int(
                (values == 0).sum()
            )

            positive_cells += int(
                (values == 1).sum()
            )

        candidate_label_tables.append({
            "path":
                str(
                    path.relative_to(
                        PROJECT_DIR
                    )
                ),
            "rows":
                len(df),
            "n_target_columns":
                len(existing_targets),
            "negative_label_cells":
                negative_cells,
            "positive_label_cells":
                positive_cells,
            "has_any_negative":
                negative_cells > 0
        })

candidate_label_tables = pd.DataFrame(
    candidate_label_tables
)

if not candidate_label_tables.empty:

    candidate_label_tables = (
        candidate_label_tables
        .sort_values(
            [
                "has_any_negative",
                "rows",
                "n_target_columns"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )

display(
    candidate_label_tables
)


,path,rows,n_target_columns,negative_label_cells,positive_label_cells,has_any_negative
0,data/public_v2/interim/catching_targets.csv,30,4,16,103,True
1,data/public_v2/interim/catching_static_validation_dataset.csv,2,4,0,8,False
2,data/public_v2/interim/catching_direct_validation_registry.csv,2,4,0,8,False


## 12. Intentar construir una lista de sitios negativos a recuperar

Este paso solo se ejecuta si se encuentra una tabla Catching con:

- al menos una Y Nielsen;
- al menos una clase negativa;
- una columna de identidad de sitio reconocible.

No se hace join por índice.


In [12]:
def detect_identity_column(df):

    preferred = [
        "site",
        "website",
        "prototype",
        "repo_name",
        "repository",
        "source_group",
        "name"
    ]

    lower_map = {
        col.lower():
            col
        for col in df.columns
    }

    for candidate in preferred:

        if candidate in lower_map:
            return lower_map[candidate]

    return None


negative_site_records = []

best_negative_table_path = None

if not candidate_label_tables.empty:

    candidates_with_negatives = (
        candidate_label_tables[
            candidate_label_tables[
                "has_any_negative"
            ]
        ]
    )

    for _, candidate in (
        candidates_with_negatives.iterrows()
    ):

        path = PROJECT_DIR / candidate["path"]

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        identity_col = detect_identity_column(
            df
        )

        if identity_col is None:
            continue

        best_negative_table_path = path

        for target in TARGETS:

            if target not in df.columns:
                continue

            values = pd.to_numeric(
                df[target],
                errors="coerce"
            )

            negative_rows = df[
                values == 0
            ]

            for _, neg_row in (
                negative_rows.iterrows()
            ):

                negative_site_records.append({
                    "source_file":
                        str(
                            path.relative_to(
                                PROJECT_DIR
                            )
                        ),
                    "identity_column":
                        identity_col,
                    "site_or_instance":
                        str(
                            neg_row[
                                identity_col
                            ]
                        ),
                    "target":
                        target,
                    "observed_label":
                        0
                })

        # Prefer the first table with negatives and usable identity.
        if negative_site_records:
            break

negative_site_recovery_priority = pd.DataFrame(
    negative_site_records
)

print(
    "Tabla Catching con negativos seleccionada:",
    (
        None
        if best_negative_table_path is None
        else best_negative_table_path.relative_to(
            PROJECT_DIR
        )
    )
)

display(
    negative_site_recovery_priority
)


Tabla Catching con negativos seleccionada: data/public_v2/interim/catching_targets.csv


,source_file,identity_column,site_or_instance,target,observed_label
0,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation,0
1,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H4_violation,0
2,data/public_v2/interim/catching_targets.csv,site,startbootstrap-freelancer,H4_violation,0
3,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H5_violation,0
4,data/public_v2/interim/catching_targets.csv,site,startbootstrap-resume,H5_violation,0
5,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H6_violation,0
6,data/public_v2/interim/catching_targets.csv,site,beginner-html-site-styled,H6_violation,0
7,data/public_v2/interim/catching_targets.csv,site,charity_responsive-template,H6_violation,0
8,data/public_v2/interim/catching_targets.csv,site,conference-website-template,H6_violation,0
9,data/public_v2/interim/catching_targets.csv,site,personal-website-template,H6_violation,0


## 13. Buscar estado de recuperación de repositorios para los sitios negativos

Se intenta localizar una tabla Catching con una columna `status` y una identidad reconocible.

Si no existe una relación explícita/reproducible, el notebook deja el estado como `UNKNOWN`; no lo infiere por posición.


In [13]:
def detect_status_column(df):

    candidates = [
        col
        for col in df.columns
        if "status" in col.lower()
    ]

    if not candidates:
        return None

    return candidates[0]


validation_candidates = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*.csv"):

        if "catch" not in path.name.lower():
            continue

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        identity_col = detect_identity_column(df)
        status_col = detect_status_column(df)

        if (
            identity_col is None
            or
            status_col is None
        ):
            continue

        validation_candidates.append({
            "path": path,
            "rows": len(df),
            "identity_col": identity_col,
            "status_col": status_col
        })

repo_status_table = None
repo_status_meta = None

if validation_candidates:

    validation_candidates = sorted(
        validation_candidates,
        key=lambda x: x["rows"],
        reverse=True
    )

    repo_status_meta = validation_candidates[0]

    repo_status_table = pd.read_csv(
        repo_status_meta["path"]
    )

print(
    "Tabla de status seleccionada:",
    (
        None
        if repo_status_meta is None
        else repo_status_meta["path"].relative_to(
            PROJECT_DIR
        )
    )
)


Tabla de status seleccionada: reports/tables/catching_multiple_candidate_validation.csv


In [14]:
def normalize_identity(value):

    value = str(value).strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )

    return value


negative_site_with_status = (
    negative_site_recovery_priority.copy()
)

negative_site_with_status[
    "repository_recovery_status"
] = "UNKNOWN"

if (
    not negative_site_with_status.empty
    and
    repo_status_table is not None
):

    identity_col = repo_status_meta[
        "identity_col"
    ]

    status_col = repo_status_meta[
        "status_col"
    ]

    status_lookup = {}

    for _, row in (
        repo_status_table.iterrows()
    ):

        key = normalize_identity(
            row[
                identity_col
            ]
        )

        if key:
            status_lookup[
                key
            ] = str(
                row[
                    status_col
                ]
            )

    negative_site_with_status[
        "repository_recovery_status"
    ] = [
        status_lookup.get(
            normalize_identity(value),
            "UNKNOWN"
        )
        for value in negative_site_with_status[
            "site_or_instance"
        ]
    ]

display(
    negative_site_with_status
)


,source_file,identity_column,site_or_instance,target,observed_label,repository_recovery_status
0,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation,0,NO_TESTABLE_EVIDENCE
1,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H4_violation,0,NO_TESTABLE_EVIDENCE
2,data/public_v2/interim/catching_targets.csv,site,startbootstrap-freelancer,H4_violation,0,NO_MATCH
3,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H5_violation,0,NO_TESTABLE_EVIDENCE
4,data/public_v2/interim/catching_targets.csv,site,startbootstrap-resume,H5_violation,0,NO_MATCH
5,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H6_violation,0,NO_TESTABLE_EVIDENCE
6,data/public_v2/interim/catching_targets.csv,site,beginner-html-site-styled,H6_violation,0,MODERATE_MATCH
7,data/public_v2/interim/catching_targets.csv,site,charity_responsive-template,H6_violation,0,NO_MATCH
8,data/public_v2/interim/catching_targets.csv,site,conference-website-template,H6_violation,0,UNKNOWN
9,data/public_v2/interim/catching_targets.csv,site,personal-website-template,H6_violation,0,NO_MATCH


## 14. Decisión por target

Se distinguen cuatro estados posibles:

- `CONVENTIONAL_SSL_READY`
- `PU_EXPLORATORY_ONLY`
- `RECOVER_LABELED_NEGATIVE_X`
- `NOT_TRAINABLE_WITH_CURRENT_EVIDENCE`

`PU_EXPLORATORY_ONLY` significa que la forma P+U existe, pero no que pueda reportarse como modelo validado.


In [15]:
recovery_targets = set(
    negative_site_recovery_priority[
        "target"
    ].tolist()
    if not negative_site_recovery_priority.empty
    else []
)

target_decision_records = []

for _, row in class_support.iterrows():

    target = row["target"]

    both_classes = bool(
        row[
            "both_classes_observed"
        ]
    )

    pu_row = pu_audit[
        pu_audit[
            "target"
        ]
        ==
        target
    ].iloc[0]

    pu_form = bool(
        pu_row[
            "pu_form_present"
        ]
    )

    negative_recovery_exists = (
        target in recovery_targets
    )

    if both_classes:

        status = (
            "CONVENTIONAL_SSL_CANDIDATE"
        )

        reason = (
            "Both classes are observed in L; "
            "grouped evaluation still needs audit."
        )

    elif negative_recovery_exists:

        status = (
            "RECOVER_LABELED_NEGATIVE_X"
        )

        reason = (
            "Negative labels exist in Catching "
            "artifacts, but compatible X has not "
            "yet been recovered for those rows."
        )

    elif pu_form:

        status = (
            "PU_EXPLORATORY_ONLY"
        )

        reason = (
            "P+U form exists, but no observed "
            "negative ground truth is available "
            "for binary validation."
        )

    else:

        status = (
            "NOT_TRAINABLE_WITH_CURRENT_EVIDENCE"
        )

        reason = (
            "Insufficient observed class support."
        )

    target_decision_records.append({
        "target": target,
        "n_L": int(row["n_L"]),
        "n_groups_L": int(row["n_groups_L"]),
        "n_0": int(row["n_0"]),
        "n_1": int(row["n_1"]),
        "n_U": int(
            pu_row[
                "n_unlabeled"
            ]
        ),
        "negative_recovery_exists":
            negative_recovery_exists,
        "trainability_status":
            status,
        "reason":
            reason
    })

target_trainability = pd.DataFrame(
    target_decision_records
)

display(
    target_trainability
)


,target,n_L,n_groups_L,n_0,n_1,n_U,negative_recovery_exists,trainability_status,reason
0,H1_violation,2,2,0,2,998,True,RECOVER_LABELED_NEGATIVE_X,"Negative labels exist in Catching artifacts, but compatible X has not yet been recovered for those rows."
1,H4_violation,2,2,0,2,998,True,RECOVER_LABELED_NEGATIVE_X,"Negative labels exist in Catching artifacts, but compatible X has not yet been recovered for those rows."
2,H5_violation,2,2,0,2,998,True,RECOVER_LABELED_NEGATIVE_X,"Negative labels exist in Catching artifacts, but compatible X has not yet been recovered for those rows."
3,H6_violation,2,2,0,2,998,True,RECOVER_LABELED_NEGATIVE_X,"Negative labels exist in Catching artifacts, but compatible X has not yet been recovered for those rows."


## 15. Decisión global de la ruta

El objetivo científico sigue siendo semisupervisado, pero no debe forzarse el notebook 28 si el seed etiquetado todavía carece de clase negativa.

Si existen etiquetas negativas Catching en artefactos previos, el siguiente trabajo correcto es recuperar X compatible para **esos sitios negativos** antes del baseline.

Eso sigue utilizando exclusivamente datasets existentes.


In [16]:
all_conventional_ready = bool(
    (
        target_trainability[
            "trainability_status"
        ]
        ==
        "CONVENTIONAL_SSL_CANDIDATE"
    ).all()
)

any_negative_recovery = bool(
    target_trainability[
        "negative_recovery_exists"
    ].any()
)

all_pu_form = bool(
    pu_audit[
        "pu_form_present"
    ].all()
)

if all_conventional_ready:

    global_status = (
        "PROCEED_TO_28_SUPERVISED_BASELINE"
    )

    next_action = (
        "Build grouped supervised baseline."
    )

elif any_negative_recovery:

    global_status = (
        "PAUSE_BEFORE_28_RECOVER_NEGATIVE_X"
    )

    next_action = (
        "Recover compatible X for Catching sites "
        "that already have observed negative "
        "Nielsen labels."
    )

elif all_pu_form:

    global_status = (
        "PU_RESEARCH_ONLY_NOT_VALIDATED"
    )

    next_action = (
        "Do not report a validated classifier; "
        "additional observed labels are required."
    )

else:

    global_status = (
        "NOT_TRAINABLE_WITH_CURRENT_EVIDENCE"
    )

    next_action = (
        "Recover additional observed labeled data."
    )

global_trainability_decision = pd.DataFrame([
    {
        "global_status":
            global_status,

        "conventional_supervised_ready":
            all_conventional_ready,

        "conventional_ssl_ready":
            all_conventional_ready,

        "pu_form_present_all_targets":
            all_pu_form,

        "domain_adaptation_required":
            True,

        "primary_model_ready":
            False
            if not all_conventional_ready
            else True,

        "next_action":
            next_action
    }
])

display(
    global_trainability_decision
)


,global_status,conventional_supervised_ready,conventional_ssl_ready,pu_form_present_all_targets,domain_adaptation_required,primary_model_ready,next_action
0,PAUSE_BEFORE_28_RECOVER_NEGATIVE_X,False,False,True,True,False,Recover compatible X for Catching sites that already have observed negative Nielsen labels.


## 16. Guardar auditoría

In [17]:
class_support.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_class_support_audit.csv",
    index=False
)

evaluation_feasibility.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_evaluation_feasibility.csv",
    index=False
)

pu_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_pu_audit.csv",
    index=False
)

domain_shift_descriptive.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_domain_shift_descriptive.csv",
    index=False
)

nearest_neighbor_domain_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_domain_nearest_neighbor.csv",
    index=False
)

ssl_readiness.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_ssl_readiness.csv",
    index=False
)

catching_artifact_inventory.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_catching_artifact_inventory.csv",
    index=False
)

candidate_label_tables.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_catching_label_table_candidates.csv",
    index=False
)

negative_site_recovery_priority.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_negative_site_recovery_priority.csv",
    index=False
)

negative_site_with_status.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_negative_site_recovery_status.csv",
    index=False
)

target_trainability.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_target_trainability.csv",
    index=False
)

global_trainability_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_global_trainability_decision.csv",
    index=False
)


In [18]:
expected_outputs = [
    "semisupervised_class_support_audit.csv",
    "semisupervised_evaluation_feasibility.csv",
    "semisupervised_pu_audit.csv",
    "semisupervised_domain_shift_descriptive.csv",
    "semisupervised_domain_nearest_neighbor.csv",
    "semisupervised_ssl_readiness.csv",
    "semisupervised_catching_artifact_inventory.csv",
    "semisupervised_catching_label_table_candidates.csv",
    "semisupervised_negative_site_recovery_priority.csv",
    "semisupervised_negative_site_recovery_status.csv",
    "semisupervised_target_trainability.csv",
    "semisupervised_global_trainability_decision.csv"
]

for filename in expected_outputs:

    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


semisupervised_class_support_audit.csv ✅
semisupervised_evaluation_feasibility.csv ✅
semisupervised_pu_audit.csv ✅
semisupervised_domain_shift_descriptive.csv ✅
semisupervised_domain_nearest_neighbor.csv ✅
semisupervised_ssl_readiness.csv ✅
semisupervised_catching_artifact_inventory.csv ✅
semisupervised_catching_label_table_candidates.csv ✅
semisupervised_negative_site_recovery_priority.csv ✅
semisupervised_negative_site_recovery_status.csv ✅
semisupervised_target_trainability.csv ✅
semisupervised_global_trainability_decision.csv ✅


## Punto de detención

No crear todavía `28_supervised_baseline.ipynb` de forma automática.

Hay dos posibles salidas científicas:

### A. `PROCEED_TO_28_SUPERVISED_BASELINE`

Solo si existe soporte observado de ambas clases y una evaluación agrupada defendible.

### B. `PAUSE_BEFORE_28_RECOVER_NEGATIVE_X`

Si ya existen etiquetas negativas Catching pero todavía falta X compatible, se debe realizar una recuperación **dirigida por los sitios negativos**, no seguir buscando datasets al azar.

Ese desvío sigue siendo parte de la misma estrategia:

```text
datasets existentes
→ recuperar X para negativos conocidos
→ L con 0/1
→ baseline supervisado
→ SSL con U
→ evaluación
```

Si el resultado fuera únicamente `PU_RESEARCH_ONLY_NOT_VALIDATED`, PU podría estudiarse como experimento metodológico, pero no debería presentarse como un clasificador Nielsen validado.
